# Sheath Evolution with Interactive Controls

This example demonstrates a simple sheath evolution visualization using Matplotlib and ipywidgets. Adjust the voltage and pressure sliders to explore how the sheath vector field responds. A basic Matplotlib animation shows the temporal evolution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import interact, FloatSlider

# Simple mock sheath model
def simulate_sheath(voltage, pressure):
    t = np.linspace(0, 2*np.pi, 100)
    sheath = np.sin(t * voltage) * np.exp(-pressure * t)
    x = np.cos(t)
    y = np.sin(t)
    u = -y * sheath
    v = x * sheath
    return x, y, u, v, sheath


In [ ]:
def plot_sheath(voltage=1.0, pressure=0.1):
    x, y, u, v, sheath = simulate_sheath(voltage, pressure)
    plt.figure()
    plt.quiver(x, y, u, v, sheath)
    plt.title('Sheath Vector Field')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.show()

interact(plot_sheath,
         voltage=FloatSlider(min=0.5, max=5.0, step=0.5, value=1.0, description='Voltage (kV)'),
         pressure=FloatSlider(min=0.05, max=1.0, step=0.05, value=0.1, description='Pressure (bar)'))


In [ ]:
# Matplotlib animation of sheath evolution
fig, ax = plt.subplots()
x, y, u, v, sheath = simulate_sheath(1.0, 0.1)
quiver = ax.quiver(x, y, u, v, sheath)
ax.set_title('Sheath Evolution (Pressure ramp)')
ax.set_xlabel('x')
ax.set_ylabel('y')

def update(frame):
    x, y, u, v, sheath = simulate_sheath(1.0, 0.1 + 0.05 * frame)
    quiver.set_UVC(u, v, sheath)
    return quiver,

ani = animation.FuncAnimation(fig, update, frames=20, blit=False)
plt.close(fig)
ani


In [ ]:
# Optional VTK/PyVista visualization
# If pyvista is installed, this cell will render a vector field using VTK.
try:
    import pyvista as pv

    x, y, u, v, sheath = simulate_sheath(1.0, 0.1)
    z = np.zeros_like(x)
    grid = pv.StructuredGrid(x, y, z)
    grid['vectors'] = np.column_stack((u, v, z))
    plotter = pv.Plotter()
    plotter.add_mesh(grid.arrows, scalars='vectors', lighting=False)
    plotter.show(jupyter_notebook=True)
except Exception as exc:
    print('PyVista/VTK not available:', exc)
